# Predicting the 2026 World Cup: A Bayesian MCMC Poisson Model

This interactive notebook models national team strength for the 32 teams advancing to the 2026 FIFA World Cup knockout stage.

### Mathematical Framework and Coefficient Derivation
To establish a highly reliable predictive model, goal-scoring and goal-concession rates are evaluated through a time-decayed lens. A team's true tactical strength is modeled as an amalgamation of their **historical baseline** (30% weight from the 2023–2026 qualification cycle) and their **immediate tournament form** (70% weight from the 2026 World Cup group stage).

Because football is inherently a low-scoring sport governed by a Poisson distribution, linear comparisons of goal differences often distort team capabilities. To rectify this, raw expected attack and defense rates are transformed using a natural logarithm, creating a standardized, zero-centered distribution:
- **Raw Attack Score**: Weighted average of goals scored per game.
- **Raw Defense Vulnerability**: Weighted average of goals conceded per game.
- **Attack Coefficient ($a_i$)**: Derived by taking $\log(\text{Raw Attack Score})$ and subtracting the tournament mean.
- **Defense Coefficient ($d_i$)**: Derived by taking $-\log(\text{Raw Defense Vulnerability})$ and adjusting to the tournament mean. A higher positive defense coefficient represents a stronger, more impenetrable defensive structure.

The Poisson rate $\lambda_{ij}$ for team $i$ playing against team $j$ is modeled as:
$$\log(\lambda_{ij}) = \mu + a_i - d_j$$
where $\mu$ is the baseline log-goal rate.

In [1]:
import math
import random

# Set seed for reproducible results
random.seed(42)

# BEGIN_KNOWN_KNOCKOUT_RESULTS
known_knockout_results = {
    ('Canada', 'South Africa'): {'goals': {'South Africa': 0, 'Canada': 1}, 'winner': 'Canada', 'is_penalty': False},
}
# END_KNOWN_KNOCKOUT_RESULTS

team_list = [
    "South Africa", "Canada", "Germany", "Paraguay",
    "Netherlands", "Morocco", "Brazil", "Japan",
    "France", "Sweden", "Ivory Coast", "Norway",
    "Mexico", "Ecuador", "England", "DR Congo",
    "USA", "Bosnia and Herzegovina", "Belgium", "Senegal",
    "Portugal", "Croatia", "Spain", "Austria",
    "Switzerland", "Algeria", "Argentina", "Cape Verde",
    "Colombia", "Ghana", "Australia", "Egypt"
]

# Priors based on the Advanced Quantitative Modeling Report (June 2026)
team_priors = {
    "Spain": {"elo": 2144, "raw_attack": 2.2167, "raw_defense": 0.1000, "attack_coef": 0.1869, "defense_coef": 2.0720},
    "Argentina": {"elo": 2144, "raw_attack": 2.3833, "raw_defense": 0.4000, "attack_coef": 0.2594, "defense_coef": 0.6857},
    "France": {"elo": 2123, "raw_attack": 3.1333, "raw_defense": 0.6667, "attack_coef": 0.5330, "defense_coef": 0.1749},
    "England": {"elo": 2038, "raw_attack": 2.2250, "raw_defense": 0.4667, "attack_coef": 0.1907, "defense_coef": 0.5315},
    "Brazil": {"elo": 2009, "raw_attack": 2.0333, "raw_defense": 0.5167, "attack_coef": 0.1006, "defense_coef": 0.4298},
    "Colombia": {"elo": 2004, "raw_attack": 1.4000, "raw_defense": 0.5333, "attack_coef": -0.2726, "defense_coef": 0.3980},
    "Portugal": {"elo": 1990, "raw_attack": 2.4000, "raw_defense": 0.5833, "attack_coef": 0.2664, "defense_coef": 0.3084},
    "Netherlands": {"elo": 1980, "raw_attack": 3.3458, "raw_defense": 1.0833, "attack_coef": 0.5986, "defense_coef": -0.3106},
    "Norway": {"elo": 1918, "raw_attack": 3.2542, "raw_defense": 1.8208, "attack_coef": 0.5709, "defense_coef": -0.8299},
    "Germany": {"elo": 1916, "raw_attack": 3.1333, "raw_defense": 1.0833, "attack_coef": 0.5330, "defense_coef": -0.3106},
    "Switzerland": {"elo": 1914, "raw_attack": 2.3333, "raw_defense": 0.8000, "attack_coef": 0.2382, "defense_coef": -0.0074},
    "Mexico": {"elo": 1912, "raw_attack": 1.7600, "raw_defense": 0.2400, "attack_coef": -0.0438, "defense_coef": 1.1965},
    "Japan": {"elo": 1910, "raw_attack": 2.5333, "raw_defense": 0.7900, "attack_coef": 0.3205, "defense_coef": 0.0051},
    "Croatia": {"elo": 1905, "raw_attack": 2.1417, "raw_defense": 1.3167, "attack_coef": 0.1525, "defense_coef": -0.5057},
    "Ecuador": {"elo": 1902, "raw_attack": 0.7000, "raw_defense": 0.5500, "attack_coef": -0.9658, "defense_coef": 0.3672},
    "Belgium": {"elo": 1884, "raw_attack": 2.4875, "raw_defense": 0.7292, "attack_coef": 0.3022, "defense_coef": 0.0853},
    "Morocco": {"elo": 1877, "raw_attack": 2.2250, "raw_defense": 0.7750, "attack_coef": 0.1907, "defense_coef": 0.0243},
    "Senegal": {"elo": 1842, "raw_attack": 2.5267, "raw_defense": 1.4900, "attack_coef": 0.3178, "defense_coef": -0.6294},
    "Austria": {"elo": 1841, "raw_attack": 2.2250, "raw_defense": 1.5500, "attack_coef": 0.1907, "defense_coef": -0.6688},
    "Paraguay": {"elo": 1815, "raw_attack": 0.7000, "raw_defense": 1.1000, "attack_coef": -0.9658, "defense_coef": -0.3259},
    "Australia": {"elo": 1800, "raw_attack": 0.9467, "raw_defense": 0.6767, "attack_coef": -0.6639, "defense_coef": 0.1600},
    "USA": {"elo": 1781, "raw_attack": 2.3167, "raw_defense": 1.2633, "attack_coef": 0.2310, "defense_coef": -0.4643},
    "Algeria": {"elo": 1780, "raw_attack": 1.8867, "raw_defense": 1.8733, "attack_coef": 0.0257, "defense_coef": -0.8583},
    "Canada": {"elo": 1748, "raw_attack": 2.1167, "raw_defense": 1.0000, "attack_coef": 0.1408, "defense_coef": -0.2306},
    "Ivory Coast": {"elo": 1743, "raw_attack": 1.6833, "raw_defense": 0.4667, "attack_coef": -0.0883, "defense_coef": 0.5315},
    "Sweden": {"elo": 1742, "raw_attack": 1.9333, "raw_defense": 2.1208, "attack_coef": 0.0502, "defense_coef": -0.9824},
    "Egypt": {"elo": 1742, "raw_attack": 1.7667, "raw_defense": 0.7600, "attack_coef": -0.0400, "defense_coef": 0.0438},
    "DR Congo": {"elo": 1712, "raw_attack": 1.3833, "raw_defense": 0.8800, "attack_coef": -0.2846, "defense_coef": -0.1028},
    "Bosnia and Herzegovina": {"elo": 1622, "raw_attack": 1.8042, "raw_defense": 1.6625, "attack_coef": -0.0190, "defense_coef": -0.7389},
    "Cape Verde": {"elo": 1622, "raw_attack": 0.9467, "raw_defense": 0.7067, "attack_coef": -0.6639, "defense_coef": 0.1166},
    "Ghana": {"elo": 1575, "raw_attack": 0.9167, "raw_defense": 0.7667, "attack_coef": -0.6961, "defense_coef": 0.0351},
    "South Africa": {"elo": 1575, "raw_attack": 0.9167, "raw_defense": 0.9700, "attack_coef": -0.6961, "defense_coef": -0.2001}
}


### Pedigree vs. Performance: The Elo Disconnect
A notable disconnect exists between established historical momentum (represented by the World Football Elo Ratings) and immediate tournament volatility. While Elo ratings provide a stable, slow-moving metric of historical pedigree, they fail to capture immediate tactical efficacy.

For example, Spain and Argentina share the apex with Elo ratings of **2144**, but teams like the **Netherlands** (Elo 1980) and **Norway** (Elo 1918) drastically over-index in immediate offensive output, generating staggering Raw Attack Scores of **3.35** and **3.25** respectively.

### Tactical Footprint: The Four Quadrants
Mapping the Attack Coefficient (horizontal axis) against the Defense Coefficient (vertical axis) reveals four distinct tactical quadrants:
1. **Elite, Balanced (Top-Right)**: Teams like **France** (att: 0.53, def: 0.17), **Japan**, **Portugal**, and **England** that transition seamlessly between offensive domination and defensive recovery.
2. **Glass Cannons (Bottom-Right)**: Teams like the **Netherlands** (att: 0.60, def: -0.31) and **Germany** (att: 0.53, def: -0.31) that compensate for defensive fragility with overwhelming goal-scoring volume.
3. **Defensive Juggernauts (Top-Left)**: Teams like **Spain** (def: 2.07) and **Mexico** (def: 1.20) that prioritize absolute structural rigidity and clean sheets.
4. **Attrition Specialists (Bottom-Left)**: Teams like **Ecuador**, **Paraguay**, **South Africa**, and **Ghana** that under-index in both but survive by dragging superior opponents into low-event, physical, and chaotic matches.

In [2]:
# Baseline log-goals (overall tournament mean)
mu = math.log(1.35)

def log_prior(params):
    log_p = 0.0
    sigma_coef = 0.15  # Prior SD for coefficients
    sigma_elo = 0.25   # Prior SD for Elo alignment
    for team, values in params.items():
        # 1. Coefficient Priors (from the report)
        prior_att = team_priors[team]["attack_coef"]
        prior_def = team_priors[team]["defense_coef"]
        log_p += -((values["attack"] - prior_att) ** 2) / (2 * sigma_coef ** 2)
        log_p += -((values["defense"] - prior_def) ** 2) / (2 * sigma_coef ** 2)
        
        # 2. Elo Prior: Net strength (attack + defense) aligned with Elo rating
        elo = team_priors[team]["elo"]
        expected_net_strength = (elo - 1850) / 200.0
        actual_net_strength = values["attack"] + values["defense"]
        log_p += -((actual_net_strength - expected_net_strength) ** 2) / (2 * sigma_elo ** 2)
    return log_p

# Real Group Stage Results to train the MCMC model
observed_matches = [
    ("Mexico", "South Africa", 2, 0),
    ("Canada", "Bosnia and Herzegovina", 1, 1),
    ("Switzerland", "Bosnia and Herzegovina", 4, 1),
    ("Switzerland", "Canada", 2, 1),
    ("Brazil", "Morocco", 1, 1),
    ("USA", "Paraguay", 4, 1),
    ("USA", "Australia", 2, 0),
    ("Paraguay", "Australia", 0, 0),
    ("Ivory Coast", "Ecuador", 1, 0),
    ("Germany", "Ivory Coast", 2, 1),
    ("Ecuador", "Germany", 2, 1),
    ("Netherlands", "Japan", 2, 2),
    ("Netherlands", "Sweden", 5, 1),
    ("Japan", "Sweden", 1, 1),
    ("Belgium", "Egypt", 1, 1),
    ("Spain", "Cape Verde", 0, 0),
    ("France", "Senegal", 3, 1),
    ("Norway", "Senegal", 3, 2),
    ("Norway", "France", 1, 4),
    ("Argentina", "Algeria", 3, 0),
    ("Argentina", "Austria", 2, 0),
    ("Algeria", "Austria", 3, 3),
    ("Portugal", "DR Congo", 1, 1),
    ("Colombia", "DR Congo", 1, 0),
    ("Colombia", "Portugal", 0, 0),
    ("England", "Croatia", 4, 2),
    ("England", "Ghana", 0, 0),
    ("Croatia", "Ghana", 2, 1)
]

# Add known knockout results to observed_matches for MCMC training
for key, res in known_knockout_results.items():
    t1, t2 = key
    g1 = res["goals"][t1]
    g2 = res["goals"][t2]
    observed_matches.append((t1, t2, g1, g2))

def log_likelihood(params):
    log_l = 0.0
    for team_a, team_b, goals_a, goals_b in observed_matches:
        att_a = params[team_a]["attack"]
        def_a = params[team_a]["defense"]
        att_b = params[team_b]["attack"]
        def_b = params[team_b]["defense"]
        
        lambda_a = math.exp(mu + att_a - def_b)
        lambda_b = math.exp(mu + att_b - def_a)
        
        log_l += goals_a * math.log(lambda_a) - lambda_a
        log_l += goals_b * math.log(lambda_b) - lambda_b
    return log_l

def log_posterior(params):
    return log_prior(params) + log_likelihood(params)


## Metropolis-Hastings MCMC Sampler and Match Simulation Formula

We define the posterior distribution and sample from it using a custom Metropolis-Hastings MCMC sampler. The chain is run for **5,000** iterations, and initialized at the prior coefficients.

### Match Simulation Model and Weight Partitioning
To simulate a match between Team A ($i$) and Team B ($j$), we calculate the Poisson goal-scoring rates $\lambda_{ij}$ (expected goals for Team A) and $\lambda_{ji}$ (expected goals for Team B).

The log-rate $\log(\lambda_{ij})$ is modeled by partitioning team capabilities into **53% Strength** (35% Elo rating, 18% MCMC posterior tournament coefficients) and **47% Luck** (zero-mean match-day random variance):

$$\log(\lambda_{ij}) = \mu + 0.35 \cdot E_{ij} + 0.18 \cdot M_{ij} + 0.47 \cdot L_i$$

Where:
- $\mu$ is the baseline log-goal rate of the tournament (fixed at $\log(1.35)$).
- **Elo Difference ($E_{ij}$)**: The historical pedigree factor, scaled such that 400 Elo points equals 1 unit on the log scale:
  $$E_{ij} = \frac{\text{Elo}_i - \text{Elo}_j}{400}$$
- **MCMC Coefficient Difference ($M_{ij}$)**: The tournament-specific strength factor, comparing Team A's MCMC posterior attack parameter ($a_i^{\text{mcmc}}$) against Team B's MCMC posterior defense parameter ($d_j^{\text{mcmc}}$):
  $$M_{ij} = a_i^{\text{mcmc}} - d_j^{\text{mcmc}}$$
- **Match-Day Luck ($L_i$)**: The highly volatile, match-specific luck/form factor, drawn from a zero-mean normal distribution:
  $$L_i \sim \mathcal{N}(0, 0.8)$$


In [3]:
# Metropolis-Hastings MCMC Sampler
def run_mcmc(iterations=5000):
    # Initialize parameters at their prior means
    current_params = {
        team: {
            "attack": team_priors[team]["attack_coef"], 
            "defense": team_priors[team]["defense_coef"]
        } 
        for team in team_list
    }
    current_log_post = log_posterior(current_params)
    
    chains = {team: {"attack": [], "defense": []} for team in team_list}
    proposal_std = 0.04
    accepted = 0
    
    for i in range(iterations):
        proposed_params = {
            team: {
                "attack": current_params[team]["attack"], 
                "defense": current_params[team]["defense"]
            } 
            for team in team_list
        }
        team_to_update = random.choice(team_list)
        
        proposed_params[team_to_update]["attack"] += random.normalvariate(0, proposal_std)
        proposed_params[team_to_update]["defense"] += random.normalvariate(0, proposal_std)
        
        proposed_log_post = log_posterior(proposed_params)
        
        log_alpha = proposed_log_post - current_log_post
        if math.log(random.random()) < log_alpha:
            current_params = proposed_params
            current_log_post = proposed_log_post
            accepted += 1
            
        for team in team_list:
            chains[team]["attack"].append(current_params[team]["attack"])
            chains[team]["defense"].append(current_params[team]["defense"])
            
    print(f"MCMC Chain complete. Acceptance rate: {accepted / iterations:.2%}")
    return chains

# Run MCMC
chains = run_mcmc(5000)
burn_in = 1000
posterior_indices = list(range(burn_in, 5000))
random.shuffle(posterior_indices)

def poisson_sample(lmbda):
    L = math.exp(-lmbda)
    k = 0
    p = 1.0
    while p > L:
        k += 1
        p *= random.random()
    return k - 1

def simulate_match_mcmc(team_a, team_b, chain_index):
    # Check if there is a known result
    key = tuple(sorted([team_a, team_b]))
    if key in known_knockout_results:
        res = known_knockout_results[key]
        goals_a = res["goals"][team_a]
        goals_b = res["goals"][team_b]
        winner = res["winner"]
        if res.get("is_penalty"):
            if winner == team_a:
                return goals_a, goals_b, f"{team_a} wins on Penalties"
            else:
                return goals_a, goals_b, f"{team_b} wins on Penalties"
        else:
            return goals_a, goals_b, f"{winner} wins in Normal/Extra Time"
    
    # 1. Elo difference (non-random, weight 35%)
    elo_a = team_priors[team_a]["elo"]
    elo_b = team_priors[team_b]["elo"]
    elo_diff = (elo_a - elo_b) / 400.0
    
    # 2. MCMC sampled parameters (representing updated strength, weight 18%)
    def_mult = 1.0
    att_a = chains[team_a]["attack"][chain_index]
    def_b = chains[team_b]["defense"][chain_index]
    mcmc_diff_a = att_a - def_mult * def_b
    
    att_b = chains[team_b]["attack"][chain_index]
    def_a = chains[team_a]["defense"][chain_index]
    mcmc_diff_b = att_b - def_mult * def_a
    
    # 3. True zero-mean match-day luck component (weight 47%)
    luck_a = random.normalvariate(0, 0.8)
    luck_b = random.normalvariate(0, 0.8)
    
    # Combine them: 53% Strength (35% Elo, 18% MCMC Coefficients) + 47% Luck
    lambda_a = math.exp(mu + 0.35 * elo_diff + 0.18 * mcmc_diff_a + 0.47 * luck_a)
    lambda_b = math.exp(mu - 0.35 * elo_diff + 0.18 * mcmc_diff_b + 0.47 * luck_b)
    
    goals_a = poisson_sample(lambda_a)
    goals_b = poisson_sample(lambda_b)
    
    if goals_a == goals_b:
        extra_lambda_a = lambda_a * 0.3
        extra_lambda_b = lambda_b * 0.3
        goals_a += poisson_sample(extra_lambda_a)
        goals_b += poisson_sample(extra_lambda_b)
        
        if goals_a == goals_b:
            prob_a = 0.5 + (0.35 * elo_diff + 0.18 * mcmc_diff_a + 0.47 * luck_a) * 0.1
            prob_a = max(0.3, min(0.7, prob_a))
            if random.random() < prob_a:
                return goals_a, goals_b, f"{team_a} wins on Penalties"
            else:
                return goals_a, goals_b, f"{team_b} wins on Penalties"
            
    winner = team_a if goals_a > goals_b else team_b
    return goals_a, goals_b, f"{winner} wins in Normal/Extra Time"

r32_matchups = [
    ("South Africa", "Canada"),
    ("Germany", "Paraguay"),
    ("Netherlands", "Morocco"),
    ("Brazil", "Japan"),
    ("France", "Sweden"),
    ("Ivory Coast", "Norway"),
    ("Mexico", "Ecuador"),
    ("England", "DR Congo"),
    ("USA", "Bosnia and Herzegovina"),
    ("Belgium", "Senegal"),
    ("Portugal", "Croatia"),
    ("Spain", "Austria"),
    ("Switzerland", "Algeria"),
    ("Argentina", "Cape Verde"),
    ("Colombia", "Ghana"),
    ("Australia", "Egypt")
]

print("--- ROUND OF 32 SIMULATION ---")
r16_teams = []
for i, (t1, t2) in enumerate(r32_matchups):
    idx = posterior_indices[i]
    g1, g2, outcome = simulate_match_mcmc(t1, t2, idx)
    winner = t1 if "wins" not in outcome or t1 in outcome else t2
    r16_teams.append(winner)
    print(f"Match {i+1:02d}: {t1:22} {g1} - {g2} {t2:22} | {outcome}")


MCMC Chain complete. Acceptance rate: 85.08%
--- ROUND OF 32 SIMULATION ---
Match 01: South Africa           0 - 1 Canada                 | Canada wins in Normal/Extra Time
Match 02: Germany                1 - 0 Paraguay               | Germany wins in Normal/Extra Time
Match 03: Netherlands            3 - 1 Morocco                | Netherlands wins in Normal/Extra Time
Match 04: Brazil                 2 - 1 Japan                  | Brazil wins in Normal/Extra Time
Match 05: France                 3 - 0 Sweden                 | France wins in Normal/Extra Time
Match 06: Ivory Coast            1 - 0 Norway                 | Ivory Coast wins in Normal/Extra Time
Match 07: Mexico                 0 - 2 Ecuador                | Ecuador wins in Normal/Extra Time
Match 08: England                2 - 1 DR Congo               | England wins in Normal/Extra Time
Match 09: USA                    0 - 0 Bosnia and Herzegovina | USA wins on Penalties
Match 10: Belgium                2 - 1 Senegal   

## Matchup Diagnostics: Netherlands vs. Morocco Expected Match Flow

Superimposing the coefficients of opposing teams allows us to predict the likely flow of play.

The Netherlands boasts the highest attack coefficient in the tournament (**0.5986**) but carries a fragile transition defense (**-0.3106**). Morocco combines a highly competent attack (**0.1907**) with neutral defensive stability (**0.0243**).

We expect the Netherlands to dominate possession, flooding the offensive third, while Morocco absorbs pressure in a mid-block and launches rapid counter-attacks into the vacated space behind the advanced Dutch wingbacks.

![Tactical Projection](./images/netherlands_morocco_tactical.svg)

## 2. Simulating the Knockout Stage (R16 to Final)

We now simulate the rest of the tournament: Round of 16, Quarter-finals, Semi-finals, and the Final using the MCMC posterior parameters.

In [4]:
print("--- ROUND OF 16 SIMULATION ---")
w73, w74, w75, w76, w77, w78, w79, w80, w81, w82, w83, w84, w85, w86, w87, w88 = r16_teams

r16_matches = [
    ("Match 89", w73, w75, 16),
    ("Match 90", w74, w77, 17),
    ("Match 91", w76, w78, 18),
    ("Match 92", w79, w80, 19),
    ("Match 93", w83, w84, 20),
    ("Match 94", w81, w82, 21),
    ("Match 95", w86, w88, 22),
    ("Match 96", w85, w87, 23)
]

r16_winners = {}
for name, t1, t2, post_idx in r16_matches:
    idx = posterior_indices[post_idx]
    g1, g2, outcome = simulate_match_mcmc(t1, t2, idx)
    winner = t1 if "wins" not in outcome or t1 in outcome else t2
    r16_winners[name] = winner
    print(f"{name}: {t1:22} {g1} - {g2} {t2:22} | {outcome}")

print("\n--- QUARTER-FINALS SIMULATION ---")
w89 = r16_winners["Match 89"]
w90 = r16_winners["Match 90"]
w91 = r16_winners["Match 91"]
w92 = r16_winners["Match 92"]
w93 = r16_winners["Match 93"]
w94 = r16_winners["Match 94"]
w95 = r16_winners["Match 95"]
w96 = r16_winners["Match 96"]

qf_matches = [
    ("Match 97", w89, w90, 24),
    ("Match 98", w93, w94, 25),
    ("Match 99", w91, w92, 26),
    ("Match 100", w95, w96, 27)
]

qf_winners = {}
for name, t1, t2, post_idx in qf_matches:
    idx = posterior_indices[post_idx]
    g1, g2, outcome = simulate_match_mcmc(t1, t2, idx)
    winner = t1 if "wins" not in outcome or t1 in outcome else t2
    qf_winners[name] = winner
    print(f"{name}: {t1:22} {g1} - {g2} {t2:22} | {outcome}")

print("\n--- SEMI-FINALS SIMULATION ---")
w97 = qf_winners["Match 97"]
w98 = qf_winners["Match 98"]
w99 = qf_winners["Match 99"]
w100 = qf_winners["Match 100"]

sf_matches = [
    ("Match 101", w97, w98, 28),
    ("Match 102", w99, w100, 29)
]

sf_winners = {}
sf_losers = {}
for name, t1, t2, post_idx in sf_matches:
    idx = posterior_indices[post_idx]
    g1, g2, outcome = simulate_match_mcmc(t1, t2, idx)
    winner = t1 if "wins" not in outcome or t1 in outcome else t2
    loser = t2 if winner == t1 else t1
    sf_winners[name] = winner
    sf_losers[name] = loser
    print(f"{name}: {t1:22} {g1} - {g2} {t2:22} | {outcome}")

print("\n--- THIRD-PLACE PLAYOFF ---")
l101 = sf_losers["Match 101"]
l102 = sf_losers["Match 102"]
idx = posterior_indices[30]
g1, g2, outcome = simulate_match_mcmc(l101, l102, idx)
print(f"Match 103: {l101:22} {g1} - {g2} {l102:22} | {outcome}")

print("\n--- WORLD CUP FINAL ---")
w101 = sf_winners["Match 101"]
w102 = sf_winners["Match 102"]
idx = posterior_indices[31]
g1, g2, outcome = simulate_match_mcmc(w101, w102, idx)
print(f"Match 104: {w101:22} {g1} - {g2} {w102:22} | {outcome}")


--- ROUND OF 16 SIMULATION ---
Match 89: Canada                 0 - 2 Netherlands            | Netherlands wins in Normal/Extra Time
Match 90: Germany                2 - 1 France                 | Germany wins in Normal/Extra Time
Match 91: Brazil                 0 - 2 Ivory Coast            | Ivory Coast wins in Normal/Extra Time
Match 92: Ecuador                0 - 1 England                | England wins in Normal/Extra Time
Match 93: Portugal               1 - 2 Spain                  | Spain wins in Normal/Extra Time
Match 94: USA                    2 - 1 Belgium                | USA wins in Normal/Extra Time
Match 95: Argentina              1 - 2 Australia              | Australia wins in Normal/Extra Time
Match 96: Algeria                0 - 2 Colombia               | Colombia wins in Normal/Extra Time

--- QUARTER-FINALS SIMULATION ---
Match 97: Netherlands            1 - 0 Germany                | Netherlands wins in Normal/Extra Time
Match 98: Spain                  2 - 0 USA 

## 3. Monte Carlo Tournament Simulation (100,000 Iterations)

To compile robust statistical probabilities of tournament success, we run the entire knockout stage (from the Round of 32 to the Final) **100,000** times. For each match in the tournament, we simulate it using MCMC posterior draws. We track the percentage of times each team reaches the Round of 16, Quarter-finals, Semi-finals, Final, and wins the Championship.

In [5]:
print("--- RUNNING MONTE CARLO SIMULATION (100,000 ITERATIONS) ---")

# Initialize stage counts
stage_counts = {
    team: {"R16": 0, "QF": 0, "SF": 0, "Final": 0, "Champion": 0}
    for team in team_list
}

num_simulations = 100000

for sim in range(num_simulations):
    # 1. Round of 32
    r16_winners = []
    for i, (t1, t2) in enumerate(r32_matchups):
        idx = random.choice(posterior_indices)
        g1, g2, outcome = simulate_match_mcmc(t1, t2, idx)
        winner = t1 if "wins" not in outcome or t1 in outcome else t2
        r16_winners.append(winner)
        stage_counts[winner]["R16"] += 1
        
    # 2. Round of 16
    w73, w74, w75, w76, w77, w78, w79, w80, w81, w82, w83, w84, w85, w86, w87, w88 = r16_winners
    r16_matches = [
        (w73, w75), (w74, w77), (w76, w78), (w79, w80),
        (w83, w84), (w81, w82), (w86, w88), (w85, w87)
    ]
    qf_winners = []
    for t1, t2 in r16_matches:
        idx = random.choice(posterior_indices)
        g1, g2, outcome = simulate_match_mcmc(t1, t2, idx)
        winner = t1 if "wins" not in outcome or t1 in outcome else t2
        qf_winners.append(winner)
        stage_counts[winner]["QF"] += 1
        
    # 3. Quarter-finals
    w89, w90, w91, w92, w93, w94, w95, w96 = qf_winners
    qf_matches = [
        (w89, w90), (w93, w94), (w91, w92), (w95, w96)
    ]
    sf_winners = []
    for t1, t2 in qf_matches:
        idx = random.choice(posterior_indices)
        g1, g2, outcome = simulate_match_mcmc(t1, t2, idx)
        winner = t1 if "wins" not in outcome or t1 in outcome else t2
        sf_winners.append(winner)
        stage_counts[winner]["SF"] += 1
        
    # 4. Semi-finals
    w97, w98, w99, w100 = sf_winners
    sf_matches = [
        (w97, w98), (w99, w100)
    ]
    finalists = []
    for t1, t2 in sf_matches:
        idx = random.choice(posterior_indices)
        g1, g2, outcome = simulate_match_mcmc(t1, t2, idx)
        winner = t1 if "wins" not in outcome or t1 in outcome else t2
        finalists.append(winner)
        stage_counts[winner]["Final"] += 1
        
    # 5. Final
    w101, w102 = finalists
    idx = random.choice(posterior_indices)
    g1, g2, outcome = simulate_match_mcmc(w101, w102, idx)
    champion = w101 if "wins" not in outcome or w101 in outcome else w102
    stage_counts[champion]["Champion"] += 1

# Print results in a beautiful table
print(f"{'Team':25} | {'R16 %':7} | {'QF %':7} | {'SF %':7} | {'Final %':7} | {'Champion %':10}")
print("-" * 75)
# Sort teams by Champion percentage
sorted_teams = sorted(team_list, key=lambda t: stage_counts[t]["Champion"], reverse=True)
for team in sorted_teams:
    r16_pct = stage_counts[team]["R16"] / num_simulations * 100
    qf_pct = stage_counts[team]["QF"] / num_simulations * 100
    sf_pct = stage_counts[team]["SF"] / num_simulations * 100
    fin_pct = stage_counts[team]["Final"] / num_simulations * 100
    champ_pct = stage_counts[team]["Champion"] / num_simulations * 100
    print(f"{team:25} | {r16_pct:5.1f}% | {qf_pct:5.1f}% | {sf_pct:5.1f}% | {fin_pct:5.1f}% | {champ_pct:8.1f}%")


--- RUNNING MONTE CARLO SIMULATION (100,000 ITERATIONS) ---
Team                      | R16 %   | QF %    | SF %    | Final % | Champion %
---------------------------------------------------------------------------
Spain                     |  76.8% |  52.5% |  40.4% |  27.2% |     17.4%
Argentina                 |  83.9% |  63.5% |  42.7% |  27.5% |     16.3%
France                    |  79.2% |  54.6% |  37.2% |  21.5% |     12.5%
England                   |  74.0% |  44.1% |  26.6% |  14.2% |      7.5%
Colombia                  |  77.5% |  47.2% |  23.2% |  12.4% |      6.0%
Brazil                    |  57.1% |  35.2% |  18.4% |   9.2% |      4.4%
Portugal                  |  58.5% |  24.2% |  15.6% |   8.2% |      4.1%
Netherlands               |  56.5% |  37.1% |  18.1% |   8.4% |      4.0%
Mexico                    |  55.8% |  27.6% |  14.6% |   6.9% |      3.2%
Switzerland               |  63.5% |  32.4% |  14.2% |   6.7% |      2.8%
Germany                   |  61.6% |  25.6% |